# 07 - Addestramento con Feature Aggiuntive
Esperimento con variabili anagrafiche (sesso, eta) e biochimiche (HbA1c, TSH, Creatinina, HDL, Trigliceridi) come feature aggiuntive (Sezione 3.5 della tesi).

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
import matplotlib.pyplot as plt

## Step 1: Correzione dei dati anagrafici e biochimici

In [ ]:
from lib.correct_dataset import process_patient_info, process_biochemical_parameters
process_patient_info()
process_biochemical_parameters()

## Step 2: Merge delle feature statiche con le misurazioni glicemiche

In [ ]:
from lib.add_features import load_datasets, process_all_patients, add_static_features, save_final_dataset

df_patients, df_glucose, df_biochem = load_datasets()
glucose_biochem_data = process_all_patients(df_glucose, df_biochem)
final_dataset = add_static_features(glucose_biochem_data, df_patients)
save_final_dataset(final_dataset, "data/T1DiabetesGranada/Glucose_measurements_with_static.csv")

## Step 3: Preparazione finestre scorrevoli con feature arricchite

In [ ]:
from lib.prepare_windows import prepare_static_windowed_data, save_static_splits

train_set, val_set, test_set, X_cols, y_cols = prepare_static_windowed_data(scale=True)
save_static_splits(train_set, val_set, test_set, X_cols, y_cols)

## Step 4: Addestramento XGBoost con feature aggiuntive

In [ ]:
X_train, y_train = train_set[X_cols], train_set[y_cols[0]]
X_val, y_val = val_set[X_cols], val_set[y_cols[0]]

model = xgb.XGBRegressor()
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred = model.predict(X_val)

# Rescale for metrics
from lib.data import rescale_data
val_results = pd.DataFrame({"target": y_val, "y_pred": y_pred, "Patient_ID": val_set["Patient_ID"]})
val_results["target"] = ((val_results["target"] + 1) * (400 - 40) / 2) + 40
val_results["y_pred"] = ((val_results["y_pred"] + 1) * (400 - 40) / 2) + 40

from lib.data import print_results
val_results["bgClass"] = val_results["target"].apply(
    lambda x: "Hypo" if x < 70 else ("Hyper" if x > 180 else "Normal")
)
print_results(val_results)

## Feature Importance

In [ ]:
importance = model.feature_importances_
feature_names = X_cols

lag_cols = [c for c in feature_names if "lag" in c]
bio_cols = [c for c in feature_names if c in ["HbA1c", "TSH", "Creatinine", "HDL", "Triglycerides"]]
demo_cols = [c for c in feature_names if c in ["Sex", "Age"]]

categories = {
    "Lag Glicemiche": sum(importance[feature_names.index(c)] for c in lag_cols if c in feature_names),
    "Biochimiche": sum(importance[feature_names.index(c)] for c in bio_cols if c in feature_names),
    "Demografiche": sum(importance[feature_names.index(c)] for c in demo_cols if c in feature_names),
}

total = sum(categories.values())
for k, v in categories.items():
    print(f"{k}: {v/total*100:.1f}%")

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(list(categories.keys()), list(categories.values()))
for bar, (k, v) in zip(bars, categories.items()):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2, f"({v/total*100:.1f}%)", va='center')
ax.set_xlabel("Feature Importance")
plt.tight_layout()
plt.show()